# CS781 Phase 4 — Experiment 2 (Reviewed v3)
## Model4 + LORMA + Hierarchical Head/Middle/Evidence/Tail Classification

This is a **distinct Phase-4 adaptation method** from Experiment 1.

### Backbone
Exact Phase-1 Model4 efficient decoder:
- 17 decoder blocks
- d_model = 640
- 10 query heads / 2 KV heads (GQA)
- RoPE + RMSNorm + SwiGLU
- FFN = 2048
- vocab = 24,576
- max positions = 2560

### LORMA
The original Phase-1 weights remain frozen. For each selected frozen linear response

`z = W0 x`

LORMA learns a multiplicative low-rank output update

`z' = z + (alpha/r) B A dropout(z)`

equivalent to

`W = (I + (alpha/r) BA) W0`.

This run uses **r=8, alpha=16** on all 17 blocks, targeting:
`q_proj, v_proj, o_proj, gate_proj, up_proj, down_proj`.

K remains frozen/unadapted.

### Long-document representation
Same strong leakage-safe pipeline as Exp1:
- deduplicated TRAIN = 32,305 unique cases;
- clean DEV = 994 cases (497/497);
- all DEV IDs that overlap official TEST are excluded;
- TEST labels/expert fields are ignored;
- normalization matches train/dev/test style;
- 4 views: HEAD / MIDDLE / EVIDENCE / TAIL;
- learned chunk attention + max pooling;
- threshold selection on exact Phase-4 Macro-F1.

### Output policy
Only essential files are kept:
`best_lorma_cjpe.pt`, `final_refit_lorma_cjpe.pt`,
`dev_probs.npz`, `test_probs.npz`, `history.json`,
`manifest.json`, and **one** `submission_best.zip`.

The LORMA checkpoints save **only trainable adapter/head parameters**, not a second 100M-parameter copy of the frozen Phase-1 backbone.


## v3 reviewed update — core experiment unchanged

I kept the **same core objective and methodology**:
**Phase-1 Model4 + multiplicative LORMA + Head/Middle/Evidence/Tail hierarchical classification**.

Useful robustness changes from the uploaded v2 are retained:
- non-finite micro-batches are skipped instead of killing the run;
- `history.json` is saved after every DEV evaluation;
- best checkpoint is written atomically;
- chunk-selection diagnostics and TEST heldout/refit agreement are logged;
- JSON/NPZ hygiene is improved;
- only one high-value submission ZIP is created.

Additional fixes in this reviewed version:
- **refit reproducibility is now real**: LORMA/head initialization and refit shuffling are both seeded with `SEED+1`;
- **final refit checkpoint is atomic** too;
- checkpoint metadata records seed and adaptation configuration;
- final manifest records label-free heldout/refit agreement;
- tokenizer JSON loading uses a context manager;
- stale temporary checkpoint files are safely removed before writes.

No change was made to:
LORMA rank/alpha/targets, 4-chunk representation, clean split, loss, learning-rate schedule,
threshold selection, refit-duration logic, or heldout+refit ensemble strategy.


In [1]:
# CELL 1 — imports/config
import os,re,gc,json,math,time,random,zipfile,subprocess,sys,hashlib,shutil,tempfile
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from sklearn.metrics import f1_score,precision_score,recall_score
from huggingface_hub import hf_hub_download,login

SEED=1234
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()

def save_json(obj,path,**kw):
    with open(path,"w",encoding="utf-8") as f: json.dump(obj,f,**kw)
def read_json(path):
    with open(path,encoding="utf-8") as f: return json.load(f)

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",DEVICE)
if DEVICE.type=="cuda":
    print("GPU:",torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32=True
    torch.backends.cudnn.allow_tf32=True
torch.set_float32_matmul_precision("high")

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN=None
assert HF_TOKEN,"HF_TOKEN missing in Kaggle Secrets"
login(token=HF_TOKEN)

HF_REPO="Exploration-Lab/CS-781-Capstone"
ROOT=Path("/kaggle/working/phase4_exp2_model4_lorma")
ROOT.mkdir(parents=True,exist_ok=True)

# Exact Phase-1 Model4
VOCAB_SIZE=24576
MAX_POSITIONS=2560
D_MODEL=640
N_LAYERS=17
N_HEADS=10
N_KV_HEADS=2
FFN_DIM=2048
ROPE_BASE=10000.0
RMS_EPS=1e-5
DROPOUT=0.0
EXPECTED_TOKENIZER_SHA256="9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d"

# Long-document representation
N_CHUNKS=4
CHUNK_LEN=384
CHUNK_NAMES=["head","middle","evidence","tail"]

# LORMA — multiplicative low-rank adaptation
LORMA_R=8
LORMA_ALPHA=16
LORMA_DROPOUT=0.05
LORMA_LR=2.0e-4
LORMA_TARGETS=("q_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")

# Hierarchical task head
CHUNK_PROJ_DIM=384
HEAD_DROPOUT=0.15
HEAD_LR=5.0e-4

# Optimization
NUM_EPOCHS=2
EVAL_PER_EPOCH=3
PATIENCE_EVALS=4
MICRO_BATCH_SIZE=2
GRAD_ACCUM_STEPS=8
EVAL_BATCH_SIZE=6
WARMUP_RATIO=0.06
WEIGHT_DECAY=0.01
GRAD_CLIP=1.0
LABEL_SMOOTHING=0.02
USE_AMP=True
AMP_INIT_SCALE=4096.0
NUM_WORKERS=2

# DEV operating point
THRESHOLDS=np.round(np.arange(0.20,0.801,0.005),3)
THRESH_SMOOTH=9

# Same data normalization learned from audit
CASE_MODE="auto"
COLLAPSE_WS=True

# Final refit and one high-value submission
RUN_REFIT=True
REFIT_MATCH_ORIGINAL_SCHEDULE=True
ENSEMBLE_HELDOUT_REFIT=True

# Neutral phrases only choose an evidence region; they never map to label 0/1.
EVIDENCE_CUES=[
    "appeal is allowed","appeal allowed",
    "appeal is dismissed","appeal dismissed",
    "petition is allowed","petition allowed",
    "petition is dismissed","petition dismissed",
    "we allow","we dismiss",
    "is hereby allowed","is hereby dismissed",
    "set aside","acquitted","convicted",
    "in the result","accordingly",
]

print("chunks:",N_CHUNKS,"x",CHUNK_LEN)
print("effective document batch:",MICRO_BATCH_SIZE*GRAD_ACCUM_STEPS)
print("LORMA:",LORMA_TARGETS,"r=",LORMA_R,"alpha=",LORMA_ALPHA,"dropout=",LORMA_DROPOUT)


Device: cuda
GPU: Tesla T4
chunks: 4 x 384
effective document batch: 16
LORMA: ('q_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj') r= 8 alpha= 16 dropout= 0.05


In [2]:
# CELL 2 — locate checkpoint/tokenizer
def all_files():
    for base in ["/kaggle/input","/kaggle/working"]:
        if not os.path.isdir(base): continue
        for root,_,files in os.walk(base):
            for fn in files:
                yield os.path.join(root,fn)

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(1<<20),b""):
            h.update(b)
    return h.hexdigest()

files=list(all_files())
ckpts=[p for p in files if p.lower().endswith(".pt") and "eff_full_best" in os.path.basename(p).lower()]
toks=[p for p in files if os.path.basename(p).lower()=="tokenizer.json"]

assert ckpts,"Attach Phase-1 Model4 eff_full_best*.pt"
assert toks,"Attach tokenizer.json"

PHASE1_CKPT=sorted(ckpts,key=lambda p:(sum(w in p.lower() for w in ["model4","phase1","efficient","eff_full_best"]),p),reverse=True)[0]
exact=[p for p in toks if sha256_file(p)==EXPECTED_TOKENIZER_SHA256]
assert exact,"Exact Phase-1 tokenizer not found"
TOKENIZER_FILE=exact[0]

print("checkpoint candidates:",len(ckpts))
for _p in ckpts: print("   -",_p)
if len(ckpts)>1: print("⚠️ several eval_full_best-style checkpoints found; verify the chosen one is Model4 (strict load below also checks).")
print("checkpoint:",PHASE1_CKPT)
print("tokenizer:",TOKENIZER_FILE)
print("SHA:",sha256_file(TOKENIZER_FILE))


checkpoint candidates: 1
   - /kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt
checkpoint: /kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt
tokenizer: /kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json
SHA: 9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d


In [3]:
# CELL 3 — exact LegalBPE loader
BPE_SPECIALS=["<pad>","<unk>","<s>","</s>"]
class LegalBPE:
    def __init__(self,vocab_size=5000):
        self.vocab_size=int(vocab_size); self.merges=[]
        self.itos=list(BPE_SPECIALS); self.stoi={t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes={}
        for b in range(256):
            s=f"{b:02x}"; self.symbol_bytes[s]=bytes([b]); self.stoi[s]=len(self.itos); self.itos.append(s)
        self._reset_encode_cache()
    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)
    @staticmethod
    def word_to_symbols(word): return [f"{b:02x}" for b in word.encode("utf-8")]
    def _reset_encode_cache(self):
        self._ranks={(l,r):i for i,(l,r) in enumerate(self.merges)}
        self._word_cache={}; self._cache_nmerges=len(self.merges)
    def encode_word(self,word):
        if self._cache_nmerges!=len(self.merges): self._reset_encode_cache()
        hit=self._word_cache.get(word)
        if hit is not None: return list(hit)
        symbols=self.word_to_symbols(word)
        while len(symbols)>1:
            best=None; rank=None
            for j in range(len(symbols)-1):
                rr=self._ranks.get((symbols[j],symbols[j+1]))
                if rr is not None and (rank is None or rr<rank):
                    best=(symbols[j],symbols[j+1]); rank=rr
            if best is None: break
            l,r=best; merged=[]; j=0
            while j<len(symbols):
                if j+1<len(symbols) and symbols[j]==l and symbols[j+1]==r:
                    merged.append(l+r); j+=2
                else:
                    merged.append(symbols[j]); j+=1
            symbols=merged
        if len(self._word_cache)>=250000: self._word_cache.clear()
        self._word_cache[word]=tuple(symbols)
        return symbols
    def encode(self,text):
        out=[]; unk=self.unk_token_id
        for unit in re.findall(r"\S+|\s+",str(text)):
            for s in self.encode_word(unit): out.append(self.stoi.get(s,unk))
        return out
    def decode(self,ids):
        buf=bytearray()
        for idx in ids:
            tok=self.itos[int(idx)]
            if tok not in BPE_SPECIALS: buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8",errors="replace")
    @classmethod
    def load_json(cls,path):
        with open(path,encoding="utf-8") as f:
            obj=json.load(f)
        tok=cls(obj["vocab_size"])
        tok.merges=[list(x) for x in obj["merges"]]
        tok.itos=list(obj["itos"]); tok.stoi={t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size=len(tok.itos); tok.symbol_bytes={}
        for b in range(256):
            s=f"{b:02x}"; tok.symbol_bytes[s]=bytes([b])
        for l,r in tok.merges: tok.symbol_bytes[l+r]=tok.symbol_bytes[l]+tok.symbol_bytes[r]
        tok._reset_encode_cache(); return tok

tokenizer=LegalBPE.load_json(TOKENIZER_FILE)
assert len(tokenizer)==VOCAB_SIZE
probe="IN THE SUPREME COURT OF INDIA\nSection 14: ₹100 — legal test."
assert tokenizer.decode(tokenizer.encode(probe))==probe
print("Tokenizer PASS:",len(tokenizer))


Tokenizer PASS: 24576


In [4]:
# CELL 4 — download Phase4 + build clean splits (+ train/test text normalisation)
FNS={
 "train":"Phase_4_CJPE/Train/train.json",
 "dev":"Phase_4_CJPE/Train/dev.json",
 "test":"Phase_4_CJPE/Test/test.json",
}
paths={k:hf_hub_download(repo_id=HF_REPO,repo_type="dataset",filename=v,token=HF_TOKEN) for k,v in FNS.items()}
train_raw=read_json(paths["train"])
dev_raw=read_json(paths["dev"])
test_raw=read_json(paths["test"])

def dedup_labeled(rows,name):
    seen={}; conflicts=set(); text_diff=0
    for r in rows:
        cid=str(r["id"]); y=int(r["label"]); txt=str(r["text"])
        assert y in (0,1)
        if cid in seen:
            if seen[cid]["label"]!=y: conflicts.add(cid)
            elif seen[cid]["text"]!=txt:
                text_diff+=1
                if len(txt)>len(seen[cid]["text"]): seen[cid]["text"]=txt   # keep the more complete version
        else:
            seen[cid]={"id":cid,"text":txt,"label":y}
    for c in conflicts: seen.pop(c,None)
    print(f"{name}: rows={len(rows)} unique={len(seen)+len(conflicts)} | same-ID different-text={text_diff} | label-conflict IDs dropped={len(conflicts)}")
    return list(seen.values())

train_records=dedup_labeled(train_raw,"train")
dev_unique=dedup_labeled(dev_raw,"dev")

# TEST: deliberately copy only id/text. Never read/use test label.
test_records=[{"id":str(r["id"]),"text":str(r["text"])} for r in test_raw]
test_ids={r["id"] for r in test_records}
assert len(test_ids)==len(test_records)

clean_dev=[r for r in dev_unique if r["id"] not in test_ids]
overlap_count=sum(r["id"] in test_ids for r in dev_unique)

if len(train_records)!=32305: print("⚠️ train unique count differs from audit:",len(train_records))
assert len(clean_dev)==994
assert len(test_records)==1517
assert overlap_count==1517
assert not ({r["id"] for r in train_records}&test_ids)
assert not ({r["id"] for r in clean_dev}&test_ids)

# ---- text style diagnostics + normalisation (train looked mixed-case, dev/test lowercase) ----
def upper_ratio(t):
    t=t[:20000]; letters=sum(c.isalpha() for c in t); up=sum(c.isupper() for c in t)
    return up/max(letters,1)
_rng=random.Random(SEED)
def sample_ratio(rows,k=1500):
    idx=_rng.sample(range(len(rows)),min(k,len(rows)))
    return np.array([upper_ratio(rows[i]["text"]) for i in idx])
ur_tr,ur_dv,ur_te=sample_ratio(train_records),sample_ratio(clean_dev),sample_ratio(test_records)
print(f"uppercase-letter ratio  train median={np.median(ur_tr):.4f} (>2% in {100*np.mean(ur_tr>0.02):.1f}% of docs)"
      f" | clean-dev median={np.median(ur_dv):.4f} | test median={np.median(ur_te):.4f}")
print("newline-containing docs  train: {:.1f}% | dev: {:.1f}% | test: {:.1f}%".format(
    *[100*np.mean(["\n" in r["text"] for r in rows[:2000]]) for rows in (train_records,clean_dev,test_records)]))

LOWERCASE=(CASE_MODE=="lower") or (CASE_MODE=="auto" and float(np.median(ur_te))<0.01)
print("LOWERCASE:",LOWERCASE,"| COLLAPSE_WS:",COLLAPSE_WS)
def normalize_text(t):
    if LOWERCASE: t=t.lower()
    if COLLAPSE_WS: t=re.sub(r"\s+"," ",t).strip()
    return t
for rows in (train_records,clean_dev,test_records):
    for r in rows: r["text"]=normalize_text(r["text"])
del train_raw,dev_raw,test_raw,dev_unique; gc.collect()

print("TRAIN:",len(train_records),Counter(r["label"] for r in train_records))
print("CLEAN DEV:",len(clean_dev),Counter(r["label"] for r in clean_dev))
print("TEST:",len(test_records))
print("Excluded DEV↔TEST overlap:",overlap_count)
print("✅ TEST label/expert fields ignored.")


Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 80.3MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

train: rows=37387 unique=32305 | same-ID different-text=0 | label-conflict IDs dropped=0
dev: rows=3505 unique=2511 | same-ID different-text=0 | label-conflict IDs dropped=0
uppercase-letter ratio  train median=0.0351 (>2% in 97.1% of docs) | clean-dev median=0.0000 | test median=0.0000
newline-containing docs  train: 0.0% | dev: 100.0% | test: 98.9%
LOWERCASE: True | COLLAPSE_WS: True
TRAIN: 32305 Counter({0: 18920, 1: 13385})
CLEAN DEV: 994 Counter({0: 497, 1: 497})
TEST: 1517
Excluded DEV↔TEST overlap: 1517
✅ TEST label/expert fields ignored.


In [5]:
# CELL 5 — exact Model4 architecture
class RMSNorm(nn.Module):
    def __init__(self,dim,eps=1e-5):
        super().__init__(); self.weight=nn.Parameter(torch.ones(dim)); self.eps=eps
    def forward(self,x):
        dtype=x.dtype; x32=x.float()
        x32=x32*torch.rsqrt(x32.pow(2).mean(-1,keepdim=True)+self.eps)
        return x32.to(dtype)*self.weight

class RotaryTable(nn.Module):
    def __init__(self,head_dim,max_positions,base=10000.0):
        super().__init__()
        inv=1.0/(base**(torch.arange(0,head_dim,2,dtype=torch.float32)/head_dim))
        freqs=torch.outer(torch.arange(max_positions,dtype=torch.float32),inv)
        self.register_buffer("cos_cached",freqs.cos(),persistent=False)
        self.register_buffer("sin_cached",freqs.sin(),persistent=False)
    def forward(self,position_ids):
        return self.cos_cached[position_ids].unsqueeze(1),self.sin_cached[position_ids].unsqueeze(1)

def apply_rope(x,cos,sin):
    B,H,T,D=x.shape; cos=cos.to(x.dtype); sin=sin.to(x.dtype)
    xp=x.view(B,H,T,D//2,2); x0,x1=xp[...,0],xp[...,1]
    return torch.stack((x0*cos-x1*sin,x0*sin+x1*cos),dim=-1).flatten(-2)

class GQACausalSelfAttention(nn.Module):
    def __init__(self,d_model,n_heads,n_kv_heads,dropout):
        super().__init__(); self.n_heads=n_heads; self.n_kv_heads=n_kv_heads
        self.head_dim=d_model//n_heads; self.kv_repeat=n_heads//n_kv_heads; self.dropout=dropout
        self.q_proj=nn.Linear(d_model,n_heads*self.head_dim,bias=False)
        self.k_proj=nn.Linear(d_model,n_kv_heads*self.head_dim,bias=False)
        self.v_proj=nn.Linear(d_model,n_kv_heads*self.head_dim,bias=False)
        self.o_proj=nn.Linear(d_model,d_model,bias=False)
    def forward(self,x,cos,sin,key_valid=None):
        B,T,C=x.shape; H=self.n_heads; KV=self.n_kv_heads; D=self.head_dim
        q=self.q_proj(x).view(B,T,H,D).transpose(1,2)
        k=self.k_proj(x).view(B,T,KV,D).transpose(1,2)
        v=self.v_proj(x).view(B,T,KV,D).transpose(1,2)
        q=apply_rope(q,cos,sin); k=apply_rope(k,cos,sin)
        k=k.repeat_interleave(self.kv_repeat,dim=1); v=v.repeat_interleave(self.kv_repeat,dim=1)
        if key_valid is None:
            mask=None; causal=True
        else:
            qi=torch.arange(T,device=x.device)[:,None]
            kj=torch.arange(T,device=x.device)[None,:]
            mask=(kj<=qi)[None,None,:,:] & key_valid[:,None,None,:]
            mask=mask | (kj==qi)[None,None,:,:]
            causal=False
        y=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,dropout_p=self.dropout if self.training else 0.0,is_causal=causal)
        return self.o_proj(y.transpose(1,2).contiguous().view(B,T,C))

class SwiGLU(nn.Module):
    def __init__(self,d_model,hidden_dim,dropout):
        super().__init__()
        self.gate_proj=nn.Linear(d_model,hidden_dim,bias=False)
        self.up_proj=nn.Linear(d_model,hidden_dim,bias=False)
        self.down_proj=nn.Linear(hidden_dim,d_model,bias=False)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x))*self.up_proj(x)))

class EfficientDecoderBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm1=RMSNorm(D_MODEL,RMS_EPS)
        self.attn=GQACausalSelfAttention(D_MODEL,N_HEADS,N_KV_HEADS,DROPOUT)
        self.norm2=RMSNorm(D_MODEL,RMS_EPS)
        self.ff=SwiGLU(D_MODEL,FFN_DIM,DROPOUT)
        self.resid_drop=nn.Dropout(DROPOUT)
    def forward(self,x,cos,sin,key_valid=None):
        x=x+self.resid_drop(self.attn(self.norm1(x),cos,sin,key_valid))
        x=x+self.ff(self.norm2(x))
        return x

class EfficientDecoderLM(nn.Module):
    def __init__(self):
        super().__init__(); self.vocab_size=VOCAB_SIZE; self.d_model=D_MODEL
        self.token_emb=nn.Embedding(VOCAB_SIZE,D_MODEL)
        self.drop=nn.Dropout(DROPOUT)
        self.rope=RotaryTable(D_MODEL//N_HEADS,MAX_POSITIONS,ROPE_BASE)
        self.blocks=nn.ModuleList([EfficientDecoderBlock() for _ in range(N_LAYERS)])
        self.final_norm=RMSNorm(D_MODEL,RMS_EPS)
    def forward_hidden(self,input_ids,key_valid=None):
        B,T=input_ids.shape; assert T<=MAX_POSITIONS
        pos=torch.arange(T,device=input_ids.device)[None,:].expand(B,T)
        cos,sin=self.rope(pos); x=self.drop(self.token_emb(input_ids))
        for block in self.blocks: x=block(x,cos,sin,key_valid)
        return self.final_norm(x)

def build_backbone():
    return EfficientDecoderLM()

bb=build_backbone()
obj=torch.load(PHASE1_CKPT,map_location="cpu",weights_only=False)
state=obj["model"] if isinstance(obj,dict) and "model" in obj else obj
bb.load_state_dict(state,strict=True)
print("Strict Model4 load PASS | params:",sum(p.numel() for p in bb.parameters())/1e6,"M")


Strict Model4 load PASS | params: 99.30944 M


In [6]:
# CELL 6 — inject LORMA + hierarchical classifier
class LoRMAPlusLinear(nn.Module):
    """
    Multiplicative low-rank adaptation of a frozen nn.Linear.

        z  = W0 x
        z' = z + (alpha/r) * B A dropout(z)

    Therefore W = (I + (alpha/r) BA) W0.

    B starts at zero, so the wrapped layer is exactly the frozen Phase-1
    projection at initialization.
    """
    def __init__(self,base,r=LORMA_R,alpha=LORMA_ALPHA,dropout=LORMA_DROPOUT):
        super().__init__()
        assert isinstance(base,nn.Linear)
        assert r>0
        self.base=base
        self.r=int(r)
        self.alpha=float(alpha)
        self.scale=self.alpha/self.r
        self.lorma_dropout=nn.Dropout(float(dropout))

        d_out=base.out_features
        self.lorma_A=nn.Parameter(torch.empty(self.r,d_out))
        self.lorma_B=nn.Parameter(torch.zeros(d_out,self.r))
        nn.init.kaiming_uniform_(self.lorma_A,a=math.sqrt(5))

        for p in self.base.parameters():
            p.requires_grad=False

    def forward(self,x):
        z=self.base(x)
        delta=F.linear(F.linear(self.lorma_dropout(z),self.lorma_A),self.lorma_B)
        return z+self.scale*delta

def inject_lorma(backbone):
    # Freeze every original Phase-1 parameter.
    for p in backbone.parameters():
        p.requires_grad=False

    injected=[]
    for bi,block in enumerate(backbone.blocks):
        # Attention projections
        for name in ("q_proj","v_proj","o_proj"):
            if name in LORMA_TARGETS:
                base=getattr(block.attn,name)
                setattr(block.attn,name,LoRMAPlusLinear(base))
                injected.append(f"blocks.{bi}.attn.{name}")

        # SwiGLU projections
        for name in ("gate_proj","up_proj","down_proj"):
            if name in LORMA_TARGETS:
                base=getattr(block.ff,name)
                setattr(block.ff,name,LoRMAPlusLinear(base))
                injected.append(f"blocks.{bi}.ff.{name}")

    lp=[(n,p) for n,p in backbone.named_parameters() if "lorma_" in n]
    assert len(lp)==2*len(injected),(len(lp),2*len(injected))
    assert all(p.requires_grad for _,p in lp)
    assert all(
        torch.count_nonzero(p.detach()).item()==0
        for n,p in lp if n.endswith("lorma_B")
    )

    expected=0
    for block in backbone.blocks:
        for modname in ("q_proj","v_proj","o_proj"):
            m=getattr(block.attn,modname)
            if isinstance(m,LoRMAPlusLinear):
                expected += 2*LORMA_R*m.base.out_features
        for modname in ("gate_proj","up_proj","down_proj"):
            m=getattr(block.ff,modname)
            if isinstance(m,LoRMAPlusLinear):
                expected += 2*LORMA_R*m.base.out_features
    actual=sum(p.numel() for _,p in lp)
    assert actual==expected,(actual,expected)

    print("Injected LORMA modules:",len(injected))
    print("LORMA trainable params:",f"{actual/1e6:.3f}M")
    return backbone

class HierarchicalCJPE(nn.Module):
    def __init__(self,backbone):
        super().__init__()
        self.backbone=backbone
        self.chunk_proj=nn.Sequential(
            nn.Linear(2*D_MODEL,CHUNK_PROJ_DIM),
            nn.GELU(),
            nn.LayerNorm(CHUNK_PROJ_DIM),
            nn.Dropout(HEAD_DROPOUT),
        )
        self.chunk_pos=nn.Embedding(N_CHUNKS,CHUNK_PROJ_DIM)
        self.attn_score=nn.Sequential(
            nn.Linear(CHUNK_PROJ_DIM,128),
            nn.Tanh(),
            nn.Linear(128,1),
        )
        self.fc=nn.Sequential(
            nn.Linear(2*CHUNK_PROJ_DIM,256),
            nn.GELU(),
            nn.Dropout(HEAD_DROPOUT),
            nn.Linear(256,2),
        )

    def forward(self,input_ids,valid):
        B,K,T=input_ids.shape
        x=input_ids.view(B*K,T)
        m=valid.view(B*K,T)

        # Right padded + causal: valid positions cannot attend to future pads.
        h=self.backbone.forward_hidden(x,key_valid=None)

        mf=m.unsqueeze(-1).to(h.dtype)
        mean=(h*mf).sum(1)/mf.sum(1).clamp(min=1)
        last_idx=m.long().sum(1).clamp(min=1)-1
        last=h[torch.arange(B*K,device=h.device),last_idx]

        z=self.chunk_proj(torch.cat([last.float(),mean.float()],dim=-1)).view(B,K,-1)
        pos=torch.arange(K,device=z.device)[None,:]
        z=z+self.chunk_pos(pos)

        w=torch.softmax(self.attn_score(z).squeeze(-1),dim=-1)
        att=(z*w.unsqueeze(-1)).sum(1)
        mx=z.max(1).values
        return self.fc(torch.cat([att,mx],dim=-1)),w

bb=inject_lorma(bb)
model=HierarchicalCJPE(bb).to(DEVICE)

# Only LORMA + task head may train.
for n,p in model.backbone.named_parameters():
    if "lorma_" not in n:
        assert not p.requires_grad,n

lorma_params=sum(p.numel() for n,p in model.named_parameters() if "lorma_" in n and p.requires_grad)
head_params=sum(p.numel() for n,p in model.named_parameters() if p.requires_grad and not n.startswith("backbone."))
total=sum(p.numel() for p in model.parameters())
print(f"LORMA trainable: {lorma_params/1e6:.3f}M")
print(f"Head trainable : {head_params/1e6:.3f}M")
print(f"Total trainable: {(lorma_params+head_params)/1e6:.3f}M / {total/1e6:.3f}M")


Injected LORMA modules: 102
LORMA trainable params: 1.671M
LORMA trainable: 1.671M
Head trainable : 0.741M
Total trainable: 2.412M / 101.722M


In [7]:
# CELL 7 — tokenize/select HEAD-MIDDLE-EVIDENCE-TAIL (slice-based: no need to BPE-encode 400k-word cases)
SLICE_CHARS=CHUNK_LEN*8     # ~3k chars >> 384 BPE tokens

def _snap_forward(text,i):
    i=max(0,min(int(i),max(0,len(text)-1)))
    m=re.search(r"\s",text[i:i+400])
    return i+m.end() if m else i

def _enc(t):
    return tokenizer.encode(t)

def _window_start(text,center,width):
    s=max(0,int(center)-width//2)
    return _snap_forward(text,s)

def _last_evidence_position(text):
    low=text.lower()
    best=-1
    for cue in EVIDENCE_CUES:
        p=low.rfind(cue)
        if p>best:
            best=p
    return best

SELECT_STATS=Counter()   # diagnostics only

def select_chunks(text):
    """
    Four views of a long case:
      head / middle / evidence-or-late / tail.
    For short cases we derive all views from exact full BPE tokens.
    """
    n=len(text); W=SLICE_CHARS

    if n<=4*W:
        SELECT_STATS["short_doc"]+=1
        ids=_enc(text) or [tokenizer.unk_token_id]
        L=len(ids)

        if L<=CHUNK_LEN:
            ch=[ids,ids,ids,ids]
        else:
            def tok_window(frac):
                center=int(round(frac*(L-1)))
                s=max(0,min(L-CHUNK_LEN,center-CHUNK_LEN//2))
                return ids[s:s+CHUNK_LEN]

            # evidence view falls back to a late (75%) view for short docs.
            ch=[
                ids[:CHUNK_LEN],
                tok_window(0.50),
                tok_window(0.75),
                ids[-CHUNK_LEN:],
            ]
        est=L

    else:
        # Prefix
        head=_enc(text[:W])

        # True character-centred middle slice.
        ms=_window_start(text,n*0.50,W)
        mid=_enc(text[ms:ms+W])

        # Evidence chunk: centre around the LAST neutral disposition cue.
        # If absent, use a late-document (75%) region.
        ep=_last_evidence_position(text)
        SELECT_STATS["long_doc"]+=1; SELECT_STATS["cue_found" if ep>=0 else "cue_missing"]+=1
        ec=(ep if ep>=0 else int(n*0.75))
        es=_window_start(text,ec,W)
        evidence=_enc(text[es:es+W])

        # Suffix
        ts=_snap_forward(text,max(0,n-W))
        tail=_enc(text[ts:])

        if min(len(head),len(mid),len(evidence),len(tail))<CHUNK_LEN:
            # Rare token-density/pathological fallback: exact full tokenisation.
            SELECT_STATS["fallback_full_tokenisation"]+=1
            ids=_enc(text) or [tokenizer.unk_token_id]
            L=len(ids)
            def tok_window(frac):
                center=int(round(frac*(L-1)))
                s=max(0,min(max(0,L-CHUNK_LEN),center-CHUNK_LEN//2))
                return ids[s:s+CHUNK_LEN]
            ch=[
                ids[:CHUNK_LEN],
                tok_window(0.50),
                tok_window(0.75),
                ids[-CHUNK_LEN:],
            ]
            est=L
        else:
            ch=[
                head[:CHUNK_LEN],
                mid[:CHUNK_LEN],
                evidence[:CHUNK_LEN],
                tail[-CHUNK_LEN:],
            ]
            est=int(n*len(head)/max(W,1))  # statistics only

    return [np.asarray(x,dtype=np.uint16) for x in ch],est

class CaseDataset(Dataset):
    def __init__(self,records,labeled):
        self.records=records; self.labeled=labeled; self.chunks=[]; self.full_lens=[]
        SELECT_STATS.clear(); t0=time.time()
        for i,r in enumerate(records):
            ch,L=select_chunks(r["text"]); self.chunks.append(ch); self.full_lens.append(L)
            if (i+1)%5000==0: print(f"tokenized {i+1}/{len(records)} elapsed={time.time()-t0:.0f}s",flush=True)
        print(f"done {len(records)} in {time.time()-t0:.0f}s | est. median BPE={np.median(self.full_lens):.0f} p95={np.percentile(self.full_lens,95):.0f}")
        print("   chunk-selection stats:",dict(SELECT_STATS))
    def __len__(self): return len(self.records)
    def __getitem__(self,i):
        r=self.records[i]; d={"id":r["id"],"chunks":self.chunks[i]}
        if self.labeled: d["label"]=int(r["label"])
        return d

def collate_cases(batch):
    B=len(batch)
    ids=torch.full((B,N_CHUNKS,CHUNK_LEN),tokenizer.pad_token_id,dtype=torch.long)
    valid=torch.zeros((B,N_CHUNKS,CHUNK_LEN),dtype=torch.bool)
    for b,x in enumerate(batch):
        for k,c in enumerate(x["chunks"]):
            z=torch.from_numpy(c.astype(np.int64)); n=min(len(z),CHUNK_LEN)
            ids[b,k,:n]=z[:n]; valid[b,k,:n]=True
    out={"input_ids":ids,"valid":valid,"ids":[x["id"] for x in batch]}
    if "label" in batch[0]:
        out["labels"]=torch.tensor([x["label"] for x in batch],dtype=torch.long)
    return out

train_ds=CaseDataset(train_records,True)
dev_ds=CaseDataset(clean_dev,True)
test_ds=CaseDataset(test_records,False)

g=torch.Generator(); g.manual_seed(SEED)
_kw=dict(collate_fn=collate_cases,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=="cuda",persistent_workers=NUM_WORKERS>0)
train_loader=DataLoader(train_ds,batch_size=MICRO_BATCH_SIZE,shuffle=True,generator=g,**_kw)
dev_loader=DataLoader(dev_ds,batch_size=EVAL_BATCH_SIZE,shuffle=False,**_kw)
test_loader=DataLoader(test_ds,batch_size=EVAL_BATCH_SIZE,shuffle=False,**_kw)

print("batches:",len(train_loader),len(dev_loader),len(test_loader))

# Verify the v2 fast causal-kernel optimisation on a real padded batch.
# Valid tokens in a causal model cannot attend to future right-padding.
@torch.no_grad()
def verify_fast_causal_equivalence():
    b=next(iter(dev_loader))
    x=b["input_ids"][:1].to(DEVICE)
    v=b["valid"][:1].to(DEVICE)
    B,K,T=x.shape
    xf=x.view(B*K,T); vf=v.view(B*K,T)
    model.backbone.eval()
    with torch.amp.autocast("cuda",dtype=torch.float16,enabled=(DEVICE.type=="cuda")):
        h_mask=model.backbone.forward_hidden(xf,key_valid=vf)
        h_fast=model.backbone.forward_hidden(xf,key_valid=None)
    d=(h_mask[vf]-h_fast[vf]).abs().max().item()
    print("fast causal-kernel valid-token max |Δ|:",d)
    assert d < 5e-2,(
        "Fast causal path deviates unexpectedly; restore key_valid masking."
    )
    model.train()

verify_fast_causal_equivalence()


tokenized 5000/32305 elapsed=14s
tokenized 10000/32305 elapsed=25s
tokenized 15000/32305 elapsed=37s
tokenized 20000/32305 elapsed=50s
tokenized 25000/32305 elapsed=62s
tokenized 30000/32305 elapsed=73s
done 32305 in 77s | est. median BPE=4416 p95=20119
   chunk-selection stats: {'long_doc': 15940, 'cue_found': 12469, 'cue_missing': 3471, 'short_doc': 16365}
done 994 in 3s | est. median BPE=6232 p95=21137
   chunk-selection stats: {'long_doc': 657, 'cue_found': 480, 'cue_missing': 177, 'short_doc': 337}
done 1517 in 4s | est. median BPE=6505 p95=22861
   chunk-selection stats: {'long_doc': 1053, 'cue_found': 772, 'short_doc': 464, 'cue_missing': 281}
batches: 16153 166 253
fast causal-kernel valid-token max |Δ|: 0.0


In [8]:
# CELL 8 — optimizer, exact metric, robust threshold search
counts=Counter(r["label"] for r in train_records)
class_w=torch.tensor(
    [len(train_records)/(2*counts[i]) for i in [0,1]],
    dtype=torch.float32,device=DEVICE,
)
class_w/=class_w.mean()
print("class weights:",class_w.tolist())

def make_groups(m):
    lorma=[p for n,p in m.named_parameters() if p.requires_grad and "lorma_" in n]
    head=[p for n,p in m.named_parameters() if p.requires_grad and not n.startswith("backbone.")]
    # No weight decay on low-rank multiplicative factors; regularize task head normally.
    return [
        {"params":lorma,"lr":LORMA_LR,"weight_decay":0.0,"name":"lorma"},
        {"params":head,"lr":HEAD_LR,"weight_decay":WEIGHT_DECAY,"name":"head"},
    ]

groups=make_groups(model)
try:
    optimizer=torch.optim.AdamW(groups,fused=(DEVICE.type=="cuda"))
except Exception:
    optimizer=torch.optim.AdamW(groups)

updates_per_epoch=math.ceil(len(train_loader)/GRAD_ACCUM_STEPS)
total_updates=NUM_EPOCHS*updates_per_epoch
warmup=max(1,int(total_updates*WARMUP_RATIO))

def lr_mult(step):
    if step<warmup:
        return max(1e-8,step/warmup)
    p=(step-warmup)/max(1,total_updates-warmup)
    return 0.5*(1+math.cos(math.pi*min(max(p,0),1)))

scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_mult)
amp_enabled=USE_AMP and DEVICE.type=="cuda"
scaler=torch.amp.GradScaler(
    "cuda",enabled=amp_enabled,
    init_scale=AMP_INIT_SCALE,growth_interval=2000,
)

def macro_f1(y,p):
    return f1_score(y,p,average="macro",labels=[0,1],zero_division=0)

def search_threshold(y,prob,smooth=THRESH_SMOOTH):
    fs=np.array(
        [macro_f1(y,(prob>=t).astype(np.int64)) for t in THRESHOLDS],
        dtype=np.float64,
    )
    k=max(1,int(smooth))
    if k>1:
        if k%2==0: k+=1
        pad=k//2
        sm=np.convolve(
            np.pad(fs,(pad,pad),mode="edge"),
            np.ones(k,dtype=np.float64)/k,
            mode="valid",
        )
    else:
        sm=fs.copy()

    i=int(np.argmax(sm))
    t=float(THRESHOLDS[i])
    pred=(prob>=t).astype(np.int64)
    return float(fs[i]),t,pred,float(fs.max())

@torch.no_grad()
def predict_logits(m,loader):
    m.eval(); lgs=[]; ys=[]; ids=[]; ats=[]
    for b in loader:
        x=b["input_ids"].to(DEVICE,non_blocking=True)
        v=b["valid"].to(DEVICE,non_blocking=True)
        with torch.amp.autocast("cuda",dtype=torch.float16,enabled=amp_enabled):
            lg,a=m(x,v)
        lgs.append(lg.float().cpu().numpy())
        ats.append(a.float().cpu().numpy())
        ids.extend(b["ids"])
        if "labels" in b:
            ys.append(b["labels"].numpy())
    return np.concatenate(lgs),(np.concatenate(ys) if ys else None),ids,np.concatenate(ats)

def _p1(lg):
    lg=lg.astype(np.float64)
    lg=lg-lg.max(-1,keepdims=True)
    e=np.exp(lg)
    return (e[:,1]/e.sum(-1)).astype(np.float32)

def predict_probs(m,loader):
    lg,y,ids,att=predict_logits(m,loader)
    return _p1(lg),y,ids,att

for pg in optimizer.param_groups:
    print(pg["name"],"lr=",pg["lr"],"params=",sum(p.numel() for p in pg["params"])/1e6,"M")
print("updates/epoch:",updates_per_epoch,"total updates:",total_updates)


class weights: [0.8286643028259277, 1.1713356971740723]
lorma lr= 2e-12 params= 1.671168 M
head lr= 5.0000000000000005e-12 params= 0.740995 M
updates/epoch: 2020 total updates: 4040


In [9]:
# CELL 9 — train / select by clean DEV Macro-F1
BEST=ROOT/"best_lorma_cjpe.pt"
BEST_TMP=BEST.with_suffix(".pt.tmp")
if BEST_TMP.exists(): BEST_TMP.unlink()  # stale temp from interrupted run only

def trainable_state_dict(m):
    return {
        k:v.detach().cpu()
        for k,v in m.state_dict().items()
        if ("lorma_" in k) or k.startswith("chunk_proj.") or k.startswith("chunk_pos.") or k.startswith("attn_score.") or k.startswith("fc.")
    }

def load_trainable_state(m,state):
    missing,unexpected=m.load_state_dict(state,strict=False)
    assert not unexpected,unexpected
    # Missing keys are intentionally frozen Phase-1 backbone weights.
    bad=[k for k in missing if ("lorma_" in k) or k.startswith(("chunk_proj.","chunk_pos.","attn_score.","fc."))]
    assert not bad,bad

history=[]
best_f1=-1.0
best_thr=0.5
best_frac=None
bad=0
global_update=0
overflows=0
nonfinite=0
optimizer.zero_grad(set_to_none=True)

eval_steps=[]
for j in range(1,EVAL_PER_EPOCH+1):
    s=int(round(len(train_loader)*j/EVAL_PER_EPOCH))
    if s<len(train_loader):
        s=max(GRAD_ACCUM_STEPS,(s//GRAD_ACCUM_STEPS)*GRAD_ACCUM_STEPS)
    else:
        s=len(train_loader)
    eval_steps.append(s)
eval_steps=sorted(set(eval_steps))
print("DEV eval batches:",eval_steps)

stop=False
for epoch in range(1,NUM_EPOCHS+1):
    model.train()
    run=0.0; nr=0; ep0=time.time(); good=0

    for step,b in enumerate(train_loader,1):
        x=b["input_ids"].to(DEVICE,non_blocking=True)
        v=b["valid"].to(DEVICE,non_blocking=True)
        y=b["labels"].to(DEVICE,non_blocking=True)

        with torch.amp.autocast("cuda",dtype=torch.float16,enabled=amp_enabled):
            lg,_=model(x,v)
            raw=F.cross_entropy(
                lg.float(),y,weight=class_w,
                label_smoothing=LABEL_SMOOTHING,
            )
            loss=raw/GRAD_ACCUM_STEPS

        # A single non-finite micro-batch must not kill a multi-hour run: skip it.
        raw_val=float(raw.detach())
        if math.isfinite(raw_val):
            scaler.scale(loss).backward()
            good+=1; run+=raw_val; nr+=1
        else:
            nonfinite+=1
            print(f"⚠️ non-finite loss at epoch={epoch} batch={step}; micro-batch skipped (total {nonfinite})")

        if step%GRAD_ACCUM_STEPS==0 or step==len(train_loader):
            if good==0:
                optimizer.zero_grad(set_to_none=True)
            else:
                scaler.unscale_(optimizer)
                gn=torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad],
                    GRAD_CLIP,
                )
                finite=bool(torch.isfinite(torch.as_tensor(gn)).item())
                old=float(scaler.get_scale())
                scaler.step(optimizer); scaler.update()
                new=float(scaler.get_scale())
                optimizer.zero_grad(set_to_none=True)

                if finite and new>=old:
                    scheduler.step(); global_update+=1
                else:
                    overflows+=1
                    print(f"⚠️ AMP overflow handled batch={step} scale {old:.0f}->{new:.0f}")
            good=0

        if step%500==0 or step==len(train_loader):
            elapsed=time.time()-ep0
            eta=(elapsed/max(step,1))*(len(train_loader)-step)
            print(
                f"epoch={epoch} batch={step}/{len(train_loader)} "
                f"loss={run/max(nr,1):.4f} elapsed={elapsed/60:.1f}m "
                f"ETA_epoch={eta/60:.1f}m",
                flush=True,
            )

        if step in eval_steps:
            prob,ydev,idsdev,att=predict_probs(model,dev_loader)
            f,t,pred,grid_best=search_threshold(ydev,prob)
            row={
                "epoch":epoch,
                "epoch_frac":(epoch-1)+step/len(train_loader),
                "batch":step,
                "macro_f1":f,
                "threshold":t,
                "grid_best_macro_f1":grid_best,
                "f1_at_0.5":float(macro_f1(ydev,(prob>=0.5).astype(np.int64))),
                "train_loss":run/max(nr,1),
                "global_update":global_update,
                "amp_overflows":overflows,
                "nonfinite_skipped":nonfinite,
                "attention_mean":att.mean(0).tolist(),
            }
            history.append(row)
            save_json(history,ROOT/"history.json",indent=2)   # survives a crashed session
            print("DEV:",row)

            if f>best_f1+1e-12:
                best_f1=f; best_thr=t; best_frac=row["epoch_frac"]; bad=0
                tmp=BEST_TMP                                  # atomic write: never leave a half-written best
                torch.save({
                    "trainable_state":trainable_state_dict(model),
                    "best_macro_f1":float(best_f1),
                    "best_threshold":float(best_thr),
                    "best_epoch_frac":float(best_frac),
                    "lorma_r":LORMA_R,
                    "lorma_alpha":LORMA_ALPHA,
                    "lorma_targets":list(LORMA_TARGETS),
                    "seed":SEED,
                    "method":"multiplicative_lorma",
                },tmp)
                os.replace(tmp,BEST)
                print(f"✅ NEW BEST Macro-F1={best_f1:.6f} threshold={best_thr:.3f}")
            else:
                bad+=1

            if bad>=PATIENCE_EVALS:
                print("Early stopping.")
                stop=True
                break
            model.train()

    if stop:
        break

save_json(history,ROOT/"history.json",indent=2)
assert BEST.is_file()
print("DONE best F1/thr/frac:",best_f1,best_thr,best_frac)
print("best adapter/head checkpoint MB:",round(BEST.stat().st_size/1024**2,2))


DEV eval batches: [5384, 10768, 16153]
epoch=1 batch=500/16153 loss=0.7027 elapsed=1.7m ETA_epoch=53.8m
epoch=1 batch=1000/16153 loss=0.7035 elapsed=3.4m ETA_epoch=52.2m
epoch=1 batch=1500/16153 loss=0.7119 elapsed=5.2m ETA_epoch=50.5m
epoch=1 batch=2000/16153 loss=0.7114 elapsed=6.9m ETA_epoch=48.8m
epoch=1 batch=2500/16153 loss=0.7085 elapsed=8.6m ETA_epoch=47.0m
epoch=1 batch=3000/16153 loss=0.7046 elapsed=10.3m ETA_epoch=45.3m
epoch=1 batch=3500/16153 loss=0.6985 elapsed=12.1m ETA_epoch=43.6m
epoch=1 batch=4000/16153 loss=0.6932 elapsed=13.8m ETA_epoch=42.0m
epoch=1 batch=4500/16153 loss=0.6875 elapsed=15.5m ETA_epoch=40.2m
epoch=1 batch=5000/16153 loss=0.6841 elapsed=17.3m ETA_epoch=38.5m
DEV: {'epoch': 1, 'epoch_frac': 0.333312697331765, 'batch': 5384, 'macro_f1': 0.5752511857673409, 'threshold': 0.29, 'grid_best_macro_f1': 0.575390832196232, 'f1_at_0.5': 0.4572528436122115, 'train_loss': 0.678988375669182, 'global_update': 673, 'amp_overflows': 0, 'nonfinite_skipped': 0, 'attent

In [10]:
# CELL 10 — exact supplied Phase-4 evaluator on clean DEV
best_obj=torch.load(BEST,map_location="cpu",weights_only=False)
load_trainable_state(model,best_obj["trainable_state"])
best_thr=float(best_obj["best_threshold"])

dev_prob,dev_y,dev_ids,dev_att=predict_probs(model,dev_loader)
dev_pred=(dev_prob>=best_thr).astype(np.int64)

truth=[{"id":str(i),"gold":int(y)} for i,y in zip(dev_ids,dev_y)]
pred=[{"id":str(i),"pred":int(p)} for i,p in zip(dev_ids,dev_pred)]

EVALUATOR_CODE=r'''#!/usr/bin/env python3
import argparse,json,sys
from sklearn.metrics import f1_score,precision_score,recall_score
def read_truth(path):
    try:
        x=json.load(open(path,encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")
    if not isinstance(x,list) or not x: raise ValueError('Truth must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'gold' not in r: raise ValueError(f'Truth item {n} must contain id and gold.')
        if not isinstance(r['id'],str) or not r['id'].strip() or isinstance(r['gold'],bool) or r['gold'] not in (0,1): raise ValueError(f'Invalid truth item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate truth ID: {r["id"]}')
        out[r['id']]=r['gold']
    return out
def read_pred(path):
    x=json.load(open(path,encoding='utf-8'))
    if not isinstance(x,list) or not x: raise ValueError('Prediction must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'pred' not in r: raise ValueError(f'Prediction item {n} must contain id and pred.')
        if not isinstance(r['id'],str) or isinstance(r['pred'],bool) or r['pred'] not in (0,1): raise ValueError(f'Invalid prediction item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate prediction ID: {r["id"]}')
        out[r['id']]=r['pred']
    return out
def main():
    ap=argparse.ArgumentParser()
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True)
    a=ap.parse_args()
    t=read_truth(a.truth); p=read_pred(a.prediction)
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    yt=[t[i] for i in t]; yp=[p[i] for i in t]
    print(f'Examples: {len(t)}')
    print(f'Macro-F1: {f1_score(yt,yp,average="macro",labels=[0,1],zero_division=0):.6f}')
    print(f'Precision: {precision_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
    print(f'Recall: {recall_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e:
        print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)
'''

with tempfile.TemporaryDirectory() as td:
    tp=Path(td)/"truth.json"
    pp=Path(td)/"prediction.json"
    ep=Path(td)/"evaluate_phase4.py"
    save_json(truth,tp)
    save_json(pred,pp)
    ep.write_text(EVALUATOR_CODE,encoding="utf-8")
    res=subprocess.run(
        [sys.executable,str(ep),"--truth",str(tp),"--prediction",str(pp)],
        capture_output=True,text=True,
    )
    print(res.stdout)
    if res.stderr: print(res.stderr)
    assert res.returncode==0

print("✅ Exact Phase-4 evaluator PASS")
np.savez_compressed(
    ROOT/"dev_probs.npz",
    prob=dev_prob.astype(np.float32),
    y=dev_y.astype(np.int64),
    id=np.asarray(dev_ids,dtype=str),   # loads without allow_pickle
    threshold=np.asarray([best_thr],dtype=np.float32),
    chunk_attention=dev_att.astype(np.float32),
)


Examples: 994
Macro-F1: 0.699496
Precision: 0.736967
Recall: 0.625755

✅ Exact Phase-4 evaluator PASS


In [11]:
# CELL 11 — fresh Phase-1+LORMA helper and clean train+DEV refit
def fresh_model():
    b=build_backbone()
    o=torch.load(PHASE1_CKPT,map_location="cpu",weights_only=False)
    st=o["model"] if isinstance(o,dict) and "model" in o else o
    b.load_state_dict(st,strict=True)
    b=inject_lorma(b)
    return HierarchicalCJPE(b).to(DEVICE)

refit_model=None
if RUN_REFIT:
    all_records=train_records+clean_dev
    all_ds=torch.utils.data.ConcatDataset([train_ds,dev_ds])

    REFIT_SEED=SEED+1
    seed_everything(REFIT_SEED)
    gg=torch.Generator(); gg.manual_seed(REFIT_SEED)
    all_loader=DataLoader(
        all_ds,batch_size=MICRO_BATCH_SIZE,shuffle=True,generator=gg,
        collate_fn=collate_cases,num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type=="cuda",persistent_workers=NUM_WORKERS>0,
    )

    refit_model=fresh_model()
    rgs=make_groups(refit_model)
    try:
        ropt=torch.optim.AdamW(rgs,fused=(DEVICE.type=="cuda"))
    except Exception:
        ropt=torch.optim.AdamW(rgs)

    target_batches=max(
        GRAD_ACCUM_STEPS,
        int(round(float(best_obj["best_epoch_frac"])*len(all_loader)/GRAD_ACCUM_STEPS))*GRAD_ACCUM_STEPS,
    )
    upe_all=math.ceil(len(all_loader)/GRAD_ACCUM_STEPS)
    horizon_updates=(NUM_EPOCHS*upe_all) if REFIT_MATCH_ORIGINAL_SCHEDULE else math.ceil(target_batches/GRAD_ACCUM_STEPS)
    rw=max(1,int(horizon_updates*WARMUP_RATIO))

    def rmult(step):
        if step<rw:
            return max(1e-8,step/rw)
        p=(step-rw)/max(1,horizon_updates-rw)
        return 0.5*(1+math.cos(math.pi*min(max(p,0),1)))

    rsched=torch.optim.lr_scheduler.LambdaLR(ropt,rmult)
    rscaler=torch.amp.GradScaler(
        "cuda",enabled=amp_enabled,
        init_scale=AMP_INIT_SCALE,growth_interval=2000,
    )

    ac=Counter(r["label"] for r in all_records)
    cw=torch.tensor(
        [len(all_records)/(2*ac[i]) for i in [0,1]],
        dtype=torch.float32,device=DEVICE,
    )
    cw/=cw.mean()

    print(
        "refit target micro-batches:",target_batches,
        "| LR horizon updates:",horizon_updates,
        "| replay original schedule:",REFIT_MATCH_ORIGINAL_SCHEDULE,
    )

    ropt.zero_grad(set_to_none=True)
    seen=0; over=0; nonfin=0; good=0; t0=time.time()

    while seen<target_batches:
        refit_model.train()
        for b in all_loader:
            if seen>=target_batches: break

            x=b["input_ids"].to(DEVICE,non_blocking=True)
            v=b["valid"].to(DEVICE,non_blocking=True)
            y=b["labels"].to(DEVICE,non_blocking=True)

            with torch.amp.autocast("cuda",dtype=torch.float16,enabled=amp_enabled):
                lg,_=refit_model(x,v)
                raw=F.cross_entropy(
                    lg.float(),y,weight=cw,
                    label_smoothing=LABEL_SMOOTHING,
                )
                loss=raw/GRAD_ACCUM_STEPS

            if math.isfinite(float(raw.detach())):
                rscaler.scale(loss).backward(); good+=1
            else:
                nonfin+=1
            seen+=1

            if seen%GRAD_ACCUM_STEPS==0 or seen>=target_batches:
                if good==0:
                    ropt.zero_grad(set_to_none=True)
                else:
                    rscaler.unscale_(ropt)
                    gn=torch.nn.utils.clip_grad_norm_(
                        [p for p in refit_model.parameters() if p.requires_grad],
                        GRAD_CLIP,
                    )
                    finite=bool(torch.isfinite(torch.as_tensor(gn)).item())
                    old=float(rscaler.get_scale())
                    rscaler.step(ropt); rscaler.update()
                    new=float(rscaler.get_scale())
                    ropt.zero_grad(set_to_none=True)

                    if finite and new>=old:
                        rsched.step()
                    else:
                        over+=1
                good=0

            if seen%500==0 or seen>=target_batches:
                rate=(time.time()-t0)/seen
                print(
                    f"refit {seen}/{target_batches} "
                    f"ETA={(target_batches-seen)*rate/60:.1f}m overflow={over} nonfinite={nonfin}",
                    flush=True,
                )

    FINAL=ROOT/"final_refit_lorma_cjpe.pt"
    FINAL_TMP=FINAL.with_suffix(".pt.tmp")
    if FINAL_TMP.exists(): FINAL_TMP.unlink()
    torch.save({
        "trainable_state":trainable_state_dict(refit_model),
        "source_best_epoch_frac":float(best_obj["best_epoch_frac"]),
        "lorma_r":LORMA_R,
        "lorma_alpha":LORMA_ALPHA,
        "lorma_targets":list(LORMA_TARGETS),
        "overflows":over,
        "nonfinite_skipped":nonfin,
        "refit_target_batches":int(target_batches),
        "refit_seed":int(REFIT_SEED),
        "method":"multiplicative_lorma",
    },FINAL_TMP)
    os.replace(FINAL_TMP,FINAL)
    print("✅ refit complete | checkpoint MB:",round(FINAL.stat().st_size/1024**2,2))
else:
    print("RUN_REFIT=False")


Injected LORMA modules: 102
LORMA trainable params: 1.671M
refit target micro-batches: 22200 | LR horizon updates: 4164 | replay original schedule: True
refit 500/22200 ETA=74.9m overflow=0 nonfinite=0
refit 1000/22200 ETA=73.2m overflow=0 nonfinite=0
refit 1500/22200 ETA=71.4m overflow=0 nonfinite=0
refit 2000/22200 ETA=69.7m overflow=0 nonfinite=0
refit 2500/22200 ETA=68.0m overflow=0 nonfinite=0
refit 3000/22200 ETA=66.3m overflow=0 nonfinite=0
refit 3500/22200 ETA=64.6m overflow=0 nonfinite=0
refit 4000/22200 ETA=62.9m overflow=0 nonfinite=0
refit 4500/22200 ETA=61.1m overflow=0 nonfinite=0
refit 5000/22200 ETA=59.4m overflow=0 nonfinite=0
refit 5500/22200 ETA=57.7m overflow=0 nonfinite=0
refit 6000/22200 ETA=56.0m overflow=0 nonfinite=0
refit 6500/22200 ETA=54.2m overflow=0 nonfinite=0
refit 7000/22200 ETA=52.5m overflow=0 nonfinite=0
refit 7500/22200 ETA=50.8m overflow=0 nonfinite=0
refit 8000/22200 ETA=49.0m overflow=0 nonfinite=0
refit 8500/22200 ETA=47.3m overflow=0 nonfinite=

In [12]:
# CELL 12 — TEST inference and label-free refit calibration
def _logit(p):
    p=np.clip(np.asarray(p,dtype=np.float64),1e-6,1-1e-6)
    return np.log(p/(1-p))

held_lg,_,held_ids,held_att=predict_logits(model,test_loader)
assert held_ids==[r["id"] for r in test_records]
assert len(held_ids)==1517
held_prob=_p1(held_lg)

refit_prob=None
refit_thr=None
test_refit_agreement=None
if refit_model is not None:
    refit_lg,_,rid,_=predict_logits(refit_model,test_loader)
    assert rid==held_ids
    refit_prob=_p1(refit_lg)

    # Refit saw clean DEV, so tune nothing with DEV labels.
    # Match its positive rate to the validated heldout model without labels.
    held_rate=float((held_prob>=best_thr).mean())
    refit_thr=float(np.clip(np.quantile(refit_prob,1.0-held_rate),0.05,0.95))
    print(
        f"heldout positive rate @ {best_thr:.3f}: {held_rate:.3f} | "
        f"refit rate-matched threshold: {refit_thr:.3f}"
    )
    test_refit_agreement=float(((held_prob>=best_thr)==(refit_prob>=refit_thr)).mean())
    print(f"TEST label-free agreement heldout vs refit: {test_refit_agreement:.4f} (low values => the two models disagree a lot)")

np.savez_compressed(
    ROOT/"test_probs.npz",
    heldout=held_prob.astype(np.float32),
    refit=(refit_prob.astype(np.float32) if refit_prob is not None else np.array([],dtype=np.float32)),
    id=np.asarray(held_ids,dtype=str),
    heldout_threshold=np.asarray([best_thr],dtype=np.float32),
    refit_threshold=np.asarray([refit_thr if refit_thr is not None else np.nan],dtype=np.float32),
)


heldout positive rate @ 0.430: 0.457 | refit rate-matched threshold: 0.449
TEST label-free agreement heldout vs refit: 0.8326 (low values => the two models disagree a lot)


In [13]:
# CELL 13 — create ONE high-value Codabench submission
official_ids=[r["id"] for r in test_records]
assert len(official_ids)==len(set(official_ids))==1517

if refit_prob is not None and ENSEMBLE_HELDOUT_REFIT:
    assert refit_thr is not None
    # Exp1 showed heldout+refit ensembling was strongest.
    # Combine margins relative to each model's own operating point.
    margin=0.5*(_logit(held_prob)-_logit(np.array([best_thr]))) \
          +0.5*(_logit(refit_prob)-_logit(np.array([refit_thr])))
    labels=(margin>=0).astype(np.int64)
    submission_source="heldout+refit calibrated-margin ensemble"
else:
    labels=(held_prob>=best_thr).astype(np.int64)
    submission_source="heldout best checkpoint"

pred=[
    {"id":str(cid),"pred":int(y)}
    for cid,y in zip(official_ids,labels)
]

# Exact structural validation.
assert len(pred)==1517
ids=[]
for r in pred:
    assert set(r)=={"id","pred"}
    assert isinstance(r["id"],str) and r["id"]
    assert type(r["pred"]) is int and r["pred"] in (0,1)
    ids.append(r["id"])
assert len(ids)==len(set(ids))
assert set(ids)==set(official_ids)

zip_path=ROOT/"submission_best.zip"
with tempfile.TemporaryDirectory() as td:
    jp=Path(td)/"prediction.json"
    save_json(pred,jp,indent=2)
    with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
        z.write(jp,arcname="prediction.json")

with zipfile.ZipFile(zip_path,"r") as z:
    assert z.namelist()==["prediction.json"]

print("✅ submission_best.zip")
print("source:",submission_source)
print("class counts:",Counter(labels.tolist()))
print("positive rate:",round(float(labels.mean()),4))


✅ submission_best.zip
source: heldout+refit calibrated-margin ensemble
class counts: Counter({0: 823, 1: 694})
positive rate: 0.4575


In [14]:
# CELL 14 — compact manifest
manifest={
    "phase":4,
    "experiment":2,
    "method":"LORMA multiplicative low-rank adaptation",
    "phase1_model":"Model4 efficient GQA/RoPE/RMSNorm/SwiGLU decoder",
    "checkpoint":str(PHASE1_CKPT),
    "tokenizer_sha256":sha256_file(TOKENIZER_FILE),
    "train_unique":len(train_records),
    "clean_dev":len(clean_dev),
    "test":len(test_records),
    "excluded_dev_test_overlap":overlap_count,
    "chunks":CHUNK_NAMES,
    "chunk_len":CHUNK_LEN,
    "evidence_selector":"last neutral disposition cue; fallback late-75pct",
    "lorma_r":LORMA_R,
    "lorma_alpha":LORMA_ALPHA,
    "lorma_dropout":LORMA_DROPOUT,
    "lorma_targets":list(LORMA_TARGETS),
    "lorma_lr":LORMA_LR,
    "head_lr":HEAD_LR,
    "best_dev_macro_f1":float(best_obj["best_macro_f1"]),
    "best_threshold":float(best_obj["best_threshold"]),
    "best_epoch_frac":float(best_obj["best_epoch_frac"]),
    "lowercase":bool(LOWERCASE),
    "collapse_ws":bool(COLLAPSE_WS),
    "submission_source":submission_source,
    "refit_run":bool(refit_model is not None),
    "refit_threshold":(None if refit_thr is None else float(refit_thr)),
    "refit_target_batches":(None if refit_model is None else int(target_batches)),
    "refit_seed":(None if refit_model is None else int(REFIT_SEED)),
    "test_refit_agreement":(None if test_refit_agreement is None else float(test_refit_agreement)),
    "test_positive_rate":float(labels.mean()),
    "torch_version":torch.__version__,
    "seed":SEED,
    "test_labels_used":False,
    "overlap_dev_labels_used":False,
}
save_json(manifest,ROOT/"manifest.json",indent=2)
print(json.dumps(manifest,indent=2))

print("\nESSENTIAL OUTPUTS ONLY:")
for fn in [
    "best_lorma_cjpe.pt",
    "final_refit_lorma_cjpe.pt",
    "dev_probs.npz",
    "test_probs.npz",
    "history.json",
    "manifest.json",
    "submission_best.zip",
]:
    p=ROOT/fn
    print(("✅" if p.exists() else "—"),fn)


{
  "phase": 4,
  "experiment": 2,
  "method": "LORMA multiplicative low-rank adaptation",
  "phase1_model": "Model4 efficient GQA/RoPE/RMSNorm/SwiGLU decoder",
  "checkpoint": "/kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt",
  "tokenizer_sha256": "9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d",
  "train_unique": 32305,
  "clean_dev": 994,
  "test": 1517,
  "excluded_dev_test_overlap": 1517,
  "chunks": [
    "head",
    "middle",
    "evidence",
    "tail"
  ],
  "chunk_len": 384,
  "evidence_selector": "last neutral disposition cue; fallback late-75pct",
  "lorma_r": 8,
  "lorma_alpha": 16,
  "lorma_dropout": 0.05,
  "lorma_targets": [
    "q_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
  ],
  "lorma_lr": 0.0002,
  "head_lr": 0.0005,
  "best_dev_macro_f1": 0.6994964429347799,
  "best_threshold": 0.43,
  "best_epoch_frac": 1.333312697331765,
  "lowercase": true,
  "collapse_ws": true,
  "submission_source":

## What to keep

Only these files matter for future Phase-4 work:

- `best_lorma_cjpe.pt`
- `final_refit_lorma_cjpe.pt`
- `dev_probs.npz`
- `test_probs.npz`
- `history.json`
- `manifest.json`
- `submission_best.zip`

The `.pt` files contain only LORMA + classification-head parameters, so keep your original Phase-1 Model4 checkpoint and tokenizer as the frozen base.

No snapshot ZIPs, raw-threshold ZIPs, prediction JSON clutter, or teacher-logit export is produced here.
Teacher logits can be generated later from these checkpoints only if this model becomes a KD teacher.
